# FiftyOne 图像分割数据集可视化

本notebook使用FiftyOne框架加载图像、分割掩码和标签，创建数据集并启动交互式可视化应用。

## 1. 导入必要的库

导入 glob、json、os、shutil、pathlib、fiftyone、PIL 和 tqdm 等必要的库。

In [ ]:
import glob
import json
import cv2
import os
import os.path as osp
import shutil
from collections import defaultdict
from pathlib import Path

import fiftyone as fo
from PIL import Image
from tqdm import tqdm

## 2. 设置图像目录路径

定义图像数据所在的目录路径，使用 Path 对象处理路径。

In [ ]:
# 设置图像目录路径
image_dir = "../data/zhenyu_data/patches/supcon_data_clean/装反"

# 验证目录是否存在
image_dir_path = Path(image_dir)
if image_dir_path.exists():
    print(f"✓ 找到数据目录: {image_dir_path.absolute()}")
else:
    print(f"✗ 数据目录不存在: {image_dir_path.absolute()}")

## 3. 扫描和收集图像文件

使用 glob 模式扫描指定目录下的所有 PNG 图像文件，排除掩码文件，并生成图像路径列表。

In [ ]:
# 扫描图像文件
image_list = list(Path(image_dir).glob("*.png"))
print(f"在 {image_dir} 中找到 {len(image_list)} 个图像文件")

# 统计掩码和非掩码文件
mask_files = [img for img in image_list if str(img).endswith("_mask.png")]
image_files = [img for img in image_list if not str(img).endswith("_mask.png")]

print(f"  - 图像文件: {len(image_files)}")
print(f"  - 掩码文件: {len(mask_files)}")

## 4. 创建注解字典

遍历图像列表，配对每个图像与其对应的分割掩码和标签信息，构建注解字典。

In [ ]:
# 创建注解字典，配对图像与掩码和标签
annotations = {}
missing_masks = []

for img_path in tqdm(image_list, desc="处理图像"):
    img_path_str = str(img_path)
    
    # 跳过掩码文件
    if img_path_str.endswith("_mask.png"):
        continue
    
    # 转换为绝对路径
    img_path_abs = Path(img_path_str).absolute()
    
    # 查找对应的掩码文件
    mask_path = img_path_abs.with_name(img_path_abs.stem + "_mask.png")
    
    if not mask_path.exists():
        missing_masks.append(img_path_abs)
        continue
    
    # 获取标签（从父文件夹名）
    label = img_path_abs.parent.name
    
    annotations[str(img_path_abs)] = [label, mask_path]

print(f"\n✓ 成功创建 {len(annotations)} 个注解")
if missing_masks:
    print(f"✗ 找不到掩码文件的图像: {len(missing_masks)} 个")

## 5. 构建 FiftyOne 样本

为每个图像创建 fo.Sample 对象，添加分类标签和分割掩码信息到样本中。

In [ ]:
# 为每个图像创建 FiftyOne 样本
samples = []

for filepath, (label, mask_path) in tqdm(annotations.items(), desc="创建样本"):
    # 创建样本
    sample = fo.Sample(filepath=filepath)

    # 添加分类标签
    sample["ground_truth"] = fo.Classification(label=label)
    
    # 添加分割掩码
    mask_image = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    # sample["segmentation"] = fo.Segmentation(filepath=str(mask_path))
    sample["segmentation"] = fo.Segmentation(mask=mask_image, tags=[label])

    samples.append(sample)

print(f"✓ 成功创建 {len(samples)} 个 FiftyOne 样本")

## 6. 创建并导出数据集

使用 FiftyOne 创建数据集，将所有样本添加到数据集中。

In [ ]:
# 创建数据集
dataset_name = "zhenyu-segmentation-dataset"

# 删除已存在的同名数据集（可选）
if fo.dataset_exists(dataset_name):
    fo.delete_dataset(dataset_name)
    print(f"删除已存在的数据集: {dataset_name}")

# 创建新数据集
dataset = fo.Dataset(dataset_name)
dataset.add_samples(samples)

print(f"✓ 成功创建数据集: {dataset_name}")
print(f"  - 样本数: {len(dataset)}")
print(f"  - 数据集大小: {dataset.stats()}")

## 7. 启动 FiftyOne 应用进行可视化

启动 FiftyOne 应用程序，在交互式界面中可视化数据集、标签和分割掩码。

**功能说明：**
- 浏览所有图像和对应的分割掩码
- 查看每个图像的分类标签
- 支持搜索、过滤和排序
- 支持标签编辑和数据管理

In [ ]:
# 启动 FiftyOne 应用
print("正在启动 FiftyOne 应用...")
session = fo.launch_app(dataset, port=5151)
print("✓ FiftyOne 应用已启动, 访问地址: http://localhost:5151")